# v2 Steals — Flat Rule Backtest: Bet Every 1.5 Under

No model. No features. Just: for every player-game with a 1.5 steals line, bet the under.
Goal: quantify the raw market inefficiency found in v1.

In [ ]:
from pathlib import Path
import subprocess, sys
import numpy as np
import pandas as pd

repo_root = Path(subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip())
sys.path.insert(0, str(repo_root))
from src.nba_rebounds_modeling.duckdb_s3_creds import connect_duckdb_s3

SEASONS = ["2023-24", "2024-25", "2025-26"]
SEASON_DATE_RANGES = {
    "2023-24": ("2023-10-01", "2024-06-30"),
    "2024-25": ("2024-10-01", "2025-06-30"),
    "2025-26": ("2025-10-01", "2026-06-30"),
}

def american_to_implied_prob(odds: float) -> float:
    if pd.isna(odds): return float("nan")
    if odds < 0: return (-odds) / ((-odds) + 100.0)
    return 100.0 / (odds + 100.0)

def american_profit(odds: float) -> float:
    if odds >= 0: return odds / 100.0
    return 100.0 / (-odds)

In [ ]:
con = connect_duckdb_s3()

logs_frames = []
for season in SEASONS:
    query = f"""
        SELECT
            PLAYER_NAME,
            CAST(STL AS DOUBLE) AS STL,
            CAST(MIN AS DOUBLE) AS MIN,
            GAME_DATE
        FROM read_csv_auto('s3://nba-api-mt/player_game_logs/{season}/*.csv',
                           header=true, ignore_errors=true)
    """
    frame = con.execute(query).df()
    frame["season"] = season
    logs_frames.append(frame)

logs = pd.concat(logs_frames, ignore_index=True)
logs = logs[logs["MIN"] > 0].copy()
logs["GAME_DATE"] = pd.to_datetime(logs["GAME_DATE"], format="mixed").dt.date
logs["player_key"] = logs["PLAYER_NAME"].str.lower().str.strip()

print(f"Game logs loaded: {len(logs):,} rows across {logs['season'].nunique()} seasons")
print(logs.groupby("season")["PLAYER_NAME"].count())

In [ ]:
props_frames = []
for season in SEASONS:
    start_date, end_date = SEASON_DATE_RANGES[season]
    query = f"""
        SELECT
            player,
            CAST(prop_line AS DOUBLE) AS prop_line,
            CAST(over_odds AS DOUBLE) AS over_odds,
            CAST(under_odds AS DOUBLE) AS under_odds,
            game_time
        FROM read_csv_auto('s3://the-odds-api-mt/nba/historical_player_props/{season}/*.csv',
                           header=true, ignore_errors=true)
        WHERE market = 'player_steals'
          AND CAST(prop_line AS DOUBLE) = 1.5
          AND game_time >= '{start_date}'
          AND game_time <= '{end_date}'
    """
    frame = con.execute(query).df()
    frame["season"] = season
    props_frames.append(frame)

props_raw = pd.concat(props_frames, ignore_index=True)
props_raw["game_time"] = pd.to_datetime(props_raw["game_time"], format="mixed")
props_raw["game_date"] = props_raw["game_time"].dt.date
props_raw["player_key"] = props_raw["player"].str.lower().str.strip()

# Aggregate to consensus: one row per (player_key, game_date) using median odds
props = (
    props_raw
    .groupby(["player_key", "game_date", "season"], as_index=False)
    .agg(
        player=("player", "first"),
        prop_line=("prop_line", "first"),
        over_odds=("over_odds", "median"),
        under_odds=("under_odds", "median"),
    )
)

print(f"Props loaded (consensus): {len(props):,} rows")
print(props.groupby("season")["player_key"].count())

In [ ]:
df = props.merge(
    logs[["player_key", "GAME_DATE", "PLAYER_NAME", "STL", "MIN", "season"]],
    left_on=["player_key", "game_date"],
    right_on=["player_key", "GAME_DATE"],
    how="inner",
    suffixes=("", "_log"),
)

# Resolve season column
if "season_log" in df.columns:
    df["season"] = df["season"].fillna(df["season_log"])
    df = df.drop(columns=["season_log"], errors="ignore")

print(f"After join: {len(df):,} rows")
print(f"Props unmatched: {len(props) - len(df):,}")

# Drop low-minute games
before = len(df)
df = df[df["MIN"] >= 15].copy()
print(f"After MIN>=15 filter: {len(df):,} rows (dropped {before - len(df):,})")
print(df.groupby("season")["PLAYER_NAME"].count())

## Market Calibration

In [ ]:
df["p_over_raw"] = df["over_odds"].apply(american_to_implied_prob)
df["p_under_raw"] = df["under_odds"].apply(american_to_implied_prob)
df["p_mkt_dv"] = df["p_over_raw"] / (df["p_over_raw"] + df["p_under_raw"])  # de-vigged P(over)
df["y_over"] = (df["STL"] >= 2).astype(int)
df["y_under"] = (df["STL"] < 2).astype(int)

# Overall calibration
print("=== Overall ===")
print(f"  n={len(df):,}")
print(f"  actual over rate (P(STL>=2)):  {df['y_over'].mean():.4f}")
print(f"  de-vigged P(over):             {df['p_mkt_dv'].mean():.4f}")
print(f"  calibration gap (mkt - actual): {df['p_mkt_dv'].mean() - df['y_over'].mean():.4f}")
print()

# By season
print("=== By season ===")
print(df.groupby("season")[["y_over", "p_mkt_dv"]].mean().assign(
    gap=lambda x: x["p_mkt_dv"] - x["y_over"]
).round(4))

## Flat Rule: Bet Every 1.5 Under

In [ ]:
df["pnl"] = df.apply(
    lambda r: american_profit(r["under_odds"]) if r["y_under"] == 1 else -1.0,
    axis=1
)

# Overall
total_pnl = df["pnl"].sum()
roi = df["pnl"].mean()
hit_rate = df["y_under"].mean()
print(f"Overall: n={len(df):,}, hit_rate={hit_rate:.3f}, ROI={roi:.4f} ({roi*100:.2f}%), total_pnl={total_pnl:.1f}u")

In [ ]:
season_summary = df.groupby("season").apply(lambda g: pd.Series({
    "n_bets": len(g),
    "hit_rate": g["y_under"].mean(),
    "roi": g["pnl"].mean(),
    "total_pnl_u": g["pnl"].sum(),
    "median_under_odds": g["under_odds"].median(),
})).round(4)
print(season_summary)

In [ ]:
df["min_bucket"] = pd.cut(df["MIN"], bins=[15, 25, 35, 100], labels=["15-25", "25-35", "35+"])
min_summary = df.groupby("min_bucket", observed=True).apply(lambda g: pd.Series({
    "n_bets": len(g),
    "hit_rate": g["y_under"].mean(),
    "roi": g["pnl"].mean(),
    "total_pnl_u": g["pnl"].sum(),
})).round(4)
print(min_summary)

In [ ]:
# Split into roughly three buckets by under_odds
df["odds_tier"] = pd.cut(df["under_odds"], bins=[-300, -200, -150, -100], 
                          labels=["<-200 (heavy fav)", "-200 to -150", "-150 to -100"])
odds_summary = df.groupby("odds_tier", observed=True).apply(lambda g: pd.Series({
    "n_bets": len(g),
    "hit_rate": g["y_under"].mean(),
    "roi": g["pnl"].mean(),
    "median_odds": g["under_odds"].median(),
})).round(4)
print(odds_summary)

## Who Drives the Edge?

In [ ]:
player_summary = df.groupby("PLAYER_NAME").apply(lambda g: pd.Series({
    "n_bets": len(g),
    "hit_rate": g["y_under"].mean(),
    "roi": g["pnl"].mean(),
    "avg_stl": g["STL"].mean(),
    "avg_line": g["prop_line"].mean() if "prop_line" in g.columns else 1.5,
    "avg_min": g["MIN"].mean(),
})).reset_index()

# Filter to min 50 bets for reliability
qual = player_summary[player_summary["n_bets"] >= 50].sort_values("roi", ascending=False)
print("Top 10 players by under ROI (n>=50):")
print(qual.head(10).to_string(index=False))
print()
print("Bottom 10 players by under ROI (n>=50):")
print(qual.tail(10).to_string(index=False))

In [ ]:
# High steal players (avg STL >= 1.5) vs low steal players
df["steal_type"] = np.where(
    df.groupby("PLAYER_NAME")["STL"].transform("mean") >= 1.2,
    "high_stealer", "low_stealer"
)
print(df.groupby("steal_type").apply(lambda g: pd.Series({
    "n_bets": len(g),
    "hit_rate": g["y_under"].mean(),
    "roi": g["pnl"].mean(),
    "p_mkt_dv": g["p_mkt_dv"].mean(),
    "calibration_gap": g["p_mkt_dv"].mean() - g["y_over"].mean(),
})).round(4))

## Cumulative PnL Over Time

In [ ]:
import matplotlib.pyplot as plt

df_sorted = df.sort_values("game_date").copy()
df_sorted["cumulative_pnl"] = df_sorted["pnl"].cumsum()

fig, ax = plt.subplots(figsize=(12, 4))
for season, grp in df_sorted.groupby("season"):
    ax.plot(range(len(grp)), grp["pnl"].cumsum().values, label=season, alpha=0.8)

ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
ax.set_xlabel("Bet number (within season)")
ax.set_ylabel("Cumulative PnL (units)")
ax.set_title("Flat 1.5 Under — Cumulative PnL by Season")
ax.legend()
plt.tight_layout()
plt.show()

# Overall cumulative
fig2, ax2 = plt.subplots(figsize=(12, 4))
ax2.plot(df_sorted["cumulative_pnl"].values, color="#3b82f6", linewidth=1.2)
ax2.axhline(0, color="gray", linestyle="--", linewidth=0.8)
ax2.set_title("Flat 1.5 Under — Cumulative PnL All Seasons")
ax2.set_xlabel("Bet number (all seasons)")
ax2.set_ylabel("Cumulative PnL (units)")
plt.tight_layout()
plt.show()

## Phase 1 Audit — Odds Data Quality

In [ ]:
# Step 1: Odds distribution
print("=== under_odds ===")
print(df["under_odds"].describe())
print()
print("=== over_odds ===")
print(df["over_odds"].describe())
print()
# Step 2: Flag out-of-range rows
weird_under = df[(df["under_odds"] > -100) | (df["under_odds"] < -400)]
n_weird = len(weird_under)
print(f"Rows with under_odds outside [-400, -100]: {n_weird} ({n_weird/len(df)*100:.1f}%)")
if n_weird > 0:
    print(weird_under[["PLAYER_NAME","game_date","season","over_odds","under_odds","STL"]].head(20))


In [ ]:
# Step 3: Check for swapped odds
# On a 1.5 steals line: under should be favorite (negative), over should be positive
n_over_pos = (df["over_odds"] > 0).sum()
n_under_pos = (df["under_odds"] > 0).sum()
n_swapped = (df["over_odds"] < df["under_odds"]).sum()
print(f"Rows where over_odds is positive (expected for 1.5 line): {n_over_pos} ({n_over_pos/len(df)*100:.1f}%)")
print(f"Rows where under_odds is positive (unexpected): {n_under_pos} ({n_under_pos/len(df)*100:.1f}%)")
print(f"Rows where over_odds < under_odds (likely swapped): {n_swapped} ({n_swapped/len(df)*100:.1f}%)")
print()
# Step 4: Inspect high-ROI players
for player in ["Jalen Suggs", "Jordan Poole", "Kawhi Leonard"]:
    sub = df[df["PLAYER_NAME"].str.contains(player, case=False, na=False)]
    if len(sub) == 0:
        print(f"{player} — not found")
        continue
    roi_p = sub["pnl"].mean()
    print(f"{player} — {len(sub)} rows, ROI={roi_p:.3f}")
    print(sub[["game_date","season","STL","over_odds","under_odds","pnl"]].sort_values("pnl", ascending=False).head(5).to_string(index=False))
    print()


In [ ]:
# Step 5: Rerun with vig-based filter + near-zero odds exclusion

def american_to_implied_prob(odds):
    if odds < 0: return (-odds) / ((-odds) + 100)
    return 100 / (odds + 100)

# Compute vig per row
df["vig"] = (
    df["over_odds"].apply(american_to_implied_prob)
    + df["under_odds"].apply(american_to_implied_prob)
)

# Filter 1: vig must be in realistic range (100-120%)
# Filter 2: neither side can be near-zero (catches -0.5, -1.0, -1.5 placeholders)
df_clean = df[
    (df["vig"] >= 1.00) & (df["vig"] <= 1.20) &
    (df["over_odds"].abs() >= 5) &
    (df["under_odds"].abs() >= 5)
].copy()

n_clean = len(df_clean)
n_total = len(df)
print(f"Rows passing filter: {n_clean:,} / {n_total:,} ({n_clean/n_total*100:.1f}%)")
print(f"Dropped: {n_total - n_clean:,}")
print(f"  - vig out of range: {((df['vig'] < 1.00) | (df['vig'] > 1.20)).sum():,}")
print(f"  - near-zero odds: {((df['over_odds'].abs() < 5) | (df['under_odds'].abs() < 5)).sum():,}")
print()

roi_clean = df_clean["pnl"].mean()
hit_clean = df_clean["y_under"].mean()
pnl_clean = df_clean["pnl"].sum()
print(f"Clean overall ROI:  {roi_clean*100:.2f}%")
print(f"Clean hit rate:     {hit_clean:.4f}")
print(f"Clean total PnL:    {pnl_clean:.2f}u")
print()

clean_by_season = df_clean.groupby("season", group_keys=False).apply(lambda g: pd.Series({
    "n_bets": len(g),
    "hit_rate": round(g["y_under"].mean(), 4),
    "roi": round(g["pnl"].mean(), 4),
    "pnl_u": round(g["pnl"].sum(), 2),
}))
print("=== Clean ROI by season ===")
print(clean_by_season.to_string())
print()

df_clean["min_bucket"] = pd.cut(df_clean["MIN"], bins=[15, 25, 35, 100], labels=["15-25","25-35","35+"])
clean_by_min = df_clean.groupby("min_bucket", observed=True, group_keys=False).apply(lambda g: pd.Series({
    "n_bets": len(g),
    "hit_rate": round(g["y_under"].mean(), 4),
    "roi": round(g["pnl"].mean(), 4),
    "pnl_u": round(g["pnl"].sum(), 2),
}))
print("=== Clean ROI by minutes ===")
print(clean_by_min.to_string())
print()

# Player breakdown on clean data
clean_by_player = df_clean.groupby("PLAYER_NAME", group_keys=False).apply(lambda g: pd.Series({
    "n_bets": len(g),
    "hit_rate": round(g["y_under"].mean(), 4),
    "roi": round(g["pnl"].mean(), 4),
    "avg_stl": round(g["STL"].mean(), 2),
    "avg_min": round(g["MIN"].mean(), 1),
})).query("n_bets >= 50").sort_values("roi", ascending=False)
print("=== Top 10 players by clean ROI (n>=50) ===")
print(clean_by_player.head(10).to_string())


In [ ]:
# Cell A: Season x minutes cross-tab
# Pass/fail: 15-25 min must be positive in all 3 seasons
cross = df_clean.groupby(["season", "min_bucket"], observed=True, group_keys=False).apply(lambda g: pd.Series({
    "n_bets": len(g),
    "hit_rate": round(g["y_under"].mean(), 4),
    "roi": round(g["pnl"].mean(), 4),
    "pnl_u": round(g["pnl"].sum(), 2),
}))
print("=== ROI by season x minutes ===")
print(cross.to_string())


In [ ]:
# Cell B: Pre-game minutes proxy
# Does min_roll10 (available pre-game) identify the same population as same-game MIN < 25?
df_clean = df_clean.sort_values(["PLAYER_NAME", "game_date"]).copy()
df_clean["min_roll10"] = (
    df_clean.groupby("PLAYER_NAME")["MIN"]
    .transform(lambda x: x.shift(1).rolling(10, min_periods=3).mean())
)

print("=== Pre-game proxy vs same-game actuals ===")
for label, mask in [
    ("same-game MIN < 25 (oracle)",  df_clean["MIN"] < 25),
    ("min_roll10 < 25",              df_clean["min_roll10"] < 25),
    ("min_roll10 < 28",              df_clean["min_roll10"] < 28),
    ("min_roll10 < 22",              df_clean["min_roll10"] < 22),
]:
    sub = df_clean[mask.fillna(False)]
    if len(sub) == 0:
        print(f"{label}: no rows")
        continue
    roi = sub["pnl"].mean()
    print(f"{label}: n={len(sub):,}, roi={roi*100:.2f}%, hit={sub["y_under"].mean():.3f}")

# Also break min_roll10 < 25 by season
print()
print("=== min_roll10 < 25 by season ===")
sub_roll = df_clean[df_clean["min_roll10"].fillna(99) < 25]
by_season = sub_roll.groupby("season", group_keys=False).apply(lambda g: pd.Series({
    "n_bets": len(g),
    "hit_rate": round(g["y_under"].mean(), 4),
    "roi": round(g["pnl"].mean(), 4),
    "pnl_u": round(g["pnl"].sum(), 2),
}))
print(by_season.to_string())


In [ ]:
# Cell C: Prove the "foul trouble / blowout" hypothesis
# For games in the oracle 15-25 min bucket, compare same-game MIN to each player's
# season-average MIN. If most of these games are starters who unexpectedly played
# short, we expect season_avg_min >> game_min.

oracle_15_25 = df_clean[df_clean["MIN"] < 25].copy()

# Compute each player's season-average MIN from the full clean dataset
season_avg_min = (
    df_clean.groupby(["PLAYER_NAME", "season"])["MIN"]
    .mean()
    .reset_index()
    .rename(columns={"MIN": "season_avg_min"})
)

oracle_15_25 = oracle_15_25.merge(season_avg_min, on=["PLAYER_NAME", "season"], how="left")
oracle_15_25["min_delta"] = oracle_15_25["season_avg_min"] - oracle_15_25["MIN"]

# Bucket by season-average (not same-game) minutes
oracle_15_25["avg_min_bucket"] = pd.cut(
    oracle_15_25["season_avg_min"],
    bins=[0, 20, 25, 30, 100],
    labels=["<20 avg", "20-25 avg", "25-30 avg", "30+ avg"]
)

print("=== Oracle 15-25 min games: what kind of player normally plays these mins? ===")
print("(are these true bench players, or starters who had a short game?)")
print()
breakdown = oracle_15_25.groupby("avg_min_bucket", observed=True).agg(
    n=("pnl", "count"),
    roi=("pnl", "mean"),
    hit_rate=("y_under", "mean"),
    avg_game_min=("MIN", "mean"),
    avg_season_min=("season_avg_min", "mean"),
    avg_delta=("min_delta", "mean")
)
breakdown["roi_pct"] = (breakdown["roi"] * 100).round(1).astype(str) + "%"
breakdown["hit_rate_pct"] = (breakdown["hit_rate"] * 100).round(1).astype(str) + "%"
print(breakdown[["n", "hit_rate_pct", "roi_pct", "avg_game_min", "avg_season_min", "avg_delta"]].to_string())

print()
print("Key: 30+ avg = starters who got pulled early. <20 avg = true bench who always play short.")

# Show concrete examples: player normally plays 30+ min, but played <20 in this game
print()
print("=== Biggest surprises: season avg >30 min but played <20 in this game ===")
big_surprise = oracle_15_25[
    (oracle_15_25["season_avg_min"] > 30) & (oracle_15_25["MIN"] < 20)
].sort_values("min_delta", ascending=False)
print(f"Count: {len(big_surprise)} games")
print()
cols = ["PLAYER_NAME", "game_date", "season", "MIN", "season_avg_min", "min_delta", "STL", "y_under", "pnl"]
print(big_surprise[cols].head(25).to_string(index=False))

# ROI comparison: true bench (avg <25) vs surprise short games (avg 25+)
print()
print("=== True bench players (season avg <25) vs starters playing short (avg 25+) ===")
oracle_15_25["player_type"] = oracle_15_25["season_avg_min"].apply(
    lambda m: "true_bench (avg <25)" if m < 25 else "starter_short_game (avg 25+)"
)
print(oracle_15_25.groupby("player_type")[["y_under", "pnl"]].agg(
    count=("pnl", "count"),
    hit_rate=("y_under", "mean"),
    roi=("pnl", "mean")
).assign(
    hit_rate=lambda d: (d["hit_rate"]*100).round(1).astype(str)+"%",
    roi=lambda d: (d["roi"]*100).round(1).astype(str)+"%"
).to_string())


## Model: Pre-Game Features → Edge vs Market

In [ ]:
# Cell D: Build pre-game features for the 1.5-line model
# All features use shift(1) within player×season — no look-ahead.

# Work from df_clean (vig-filtered) which already has MIN, STL, min_roll10
dc = df_clean.sort_values(["PLAYER_NAME", "season", "game_date"]).copy()

def roll_shift(series, window):
    """Rolling mean with shift(1) — excludes current game."""
    return series.shift(1).rolling(window, min_periods=max(1, window // 2)).mean()

grp = dc.groupby(["PLAYER_NAME", "season"])

dc["stl_roll5"]        = grp["STL"].transform(lambda s: roll_shift(s, 5))
dc["stl_roll10"]       = grp["STL"].transform(lambda s: roll_shift(s, 10))
dc["stl_roll20"]       = grp["STL"].transform(lambda s: roll_shift(s, 20))
dc["min_roll10"]       = grp["MIN"].transform(lambda s: roll_shift(s, 10))
dc["min_roll20"]       = grp["MIN"].transform(lambda s: roll_shift(s, 20))

# Steal rate per minute (handle div-by-zero)
dc["stl_per_min_roll10"] = dc["stl_roll10"] / dc["min_roll10"].replace(0, np.nan)

# Games played so far this season (proxy for how well we know this player)
dc["games_played"] = grp.cumcount()

# Market-implied P(under) — de-vigged, pre-game available
dc["market_p_under"] = 1 - dc["p_mkt_dv"]  # p_mkt_dv is de-vigged P(over)

# Drop rows where rolling features are NaN (first few games of season)
FEATURES = ["stl_roll5", "stl_roll10", "stl_roll20",
            "min_roll10", "min_roll20", "stl_per_min_roll10",
            "games_played"]

dc_model = dc.dropna(subset=FEATURES).copy()
print(f"Rows after dropping NaN features: {len(dc_model):,} / {len(dc):,}")
print(f"Target (y_under) distribution: {dc_model['y_under'].value_counts().to_dict()}")
print(f"Base rate P(under): {dc_model['y_under'].mean():.3f}")
print()
print("Feature summary:")
print(dc_model[FEATURES].describe().round(3).to_string())


In [ ]:
# Cell E: Walk-forward backtest — LogReg model vs market
# Fold 1: train 2023-24 → test 2024-25
# Fold 2: train 2023-24 + 2024-25 → test 2025-26

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import brier_score_loss, log_loss

SEASONS_ORDERED = ["2023-24", "2024-25", "2025-26"]
EDGE_THRESHOLDS = [0.03, 0.05, 0.07, 0.10]

results = []

for test_idx in range(1, len(SEASONS_ORDERED)):
    train_seasons = SEASONS_ORDERED[:test_idx]
    test_season   = SEASONS_ORDERED[test_idx]

    train = dc_model[dc_model["season"].isin(train_seasons)].copy()
    test  = dc_model[dc_model["season"] == test_season].copy()

    scaler = StandardScaler()
    X_train = scaler.fit_transform(train[FEATURES])
    X_test  = scaler.transform(test[FEATURES])
    y_train = train["y_under"].values
    y_test  = test["y_under"].values

    model = LogisticRegression(max_iter=1000, C=0.5)
    model.fit(X_train, y_train)

    test = test.copy()
    test["model_p_under"] = model.predict_proba(X_test)[:, 1]
    test["edge"] = test["model_p_under"] - test["market_p_under"]

    # Brier scores
    brier_model  = brier_score_loss(y_test, test["model_p_under"])
    brier_market = brier_score_loss(y_test, test["market_p_under"])

    print(f"=== Fold: train {train_seasons} → test {test_season} ===")
    print(f"  Test n: {len(test):,}")
    print(f"  Brier model: {brier_model:.5f}  |  Brier market: {brier_market:.5f}  |  "
          f"{'MODEL WINS' if brier_model < brier_market else 'market wins'}")
    print()
    print(f"  {'Edge thresh':>12}  {'n_bets':>8}  {'hit_rate':>10}  {'ROI':>8}  {'PnL':>8}")
    print(f"  {'-'*52}")

    # Flat rule baseline on same universe
    flat_roi = test["pnl"].mean()
    print(f"  {'flat rule':>12}  {len(test):>8}  {y_test.mean()*100:>9.1f}%  {flat_roi*100:>7.1f}%  {test['pnl'].sum():>7.1f}u")

    for thresh in EDGE_THRESHOLDS:
        bets = test[test["edge"] >= thresh]
        if len(bets) == 0:
            print(f"  {thresh:>+12.0%}  {'0':>8}  {'—':>10}  {'—':>8}  {'—':>8}")
            continue
        roi  = bets["pnl"].mean()
        pnl  = bets["pnl"].sum()
        hr   = bets["y_under"].mean()
        print(f"  {thresh:>+12.0%}  {len(bets):>8}  {hr*100:>9.1f}%  {roi*100:>7.1f}%  {pnl:>7.1f}u")
        results.append({"fold": test_season, "threshold": thresh,
                        "n_bets": len(bets), "roi": roi, "hit_rate": hr})
    print()

    # Show top features by coefficient
    coef_df = pd.DataFrame({"feature": FEATURES, "coef": model.coef_[0]})
    coef_df = coef_df.reindex(coef_df["coef"].abs().sort_values(ascending=False).index)
    print(f"  Feature coefficients (→ under):")
    for _, row in coef_df.iterrows():
        print(f"    {row['feature']:30s}  {row['coef']:+.3f}")
    print()


In [ ]:
# Cell F: Calibration plot — model P(under) vs actual under rate
# Checks whether the model's probabilities are meaningful

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Re-run fold 2 predictions to get a full out-of-sample set
train2 = dc_model[dc_model["season"].isin(["2023-24", "2024-25"])].copy()
test2  = dc_model[dc_model["season"] == "2025-26"].copy()

scaler2 = StandardScaler()
X_tr2   = scaler2.fit_transform(train2[FEATURES])
X_te2   = scaler2.transform(test2[FEATURES])

model2  = LogisticRegression(max_iter=1000, C=0.5)
model2.fit(X_tr2, train2["y_under"].values)
test2["model_p_under"] = model2.predict_proba(X_te2)[:, 1]
test2["edge"]          = test2["model_p_under"] - test2["market_p_under"]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: edge distribution
ax = axes[0]
ax.hist(test2["edge"], bins=40, color="#3b82f6", edgecolor="#1e40af", alpha=0.8)
for t in [0.03, 0.05, 0.07, 0.10]:
    ax.axvline(t, color="#f59e0b", linestyle="--", linewidth=1)
ax.axvline(0, color="#ef4444", linestyle="-", linewidth=1.5)
ax.set_xlabel("Edge = model P(under) − market P(under)")
ax.set_ylabel("Count")
ax.set_title("Edge Distribution (2025-26 test set)")
ax.set_facecolor("#0f1117")
fig.patch.set_facecolor("#0f1117")
ax.tick_params(colors="#e2e8f0")
for spine in ax.spines.values(): spine.set_edgecolor("#2d3748")

# Right: calibration — model P(under) decile vs actual hit rate
ax2 = axes[1]
test2["p_decile"] = pd.qcut(test2["model_p_under"], q=10, duplicates="drop")
cal = test2.groupby("p_decile", observed=True).agg(
    mid_p=("model_p_under", "mean"),
    actual=("y_under", "mean"),
    n=("y_under", "count")
)
ax2.plot(cal["mid_p"], cal["actual"], "o-", color="#34d399", label="Model")
ax2.plot([0.5, 1.0], [0.5, 1.0], "--", color="#94a3b8", label="Perfect calibration")
ax2.set_xlabel("Model predicted P(under)")
ax2.set_ylabel("Actual under rate")
ax2.set_title("Model Calibration (2025-26)")
ax2.legend()
ax2.set_facecolor("#0f1117")
ax2.tick_params(colors="#e2e8f0")
for spine in ax2.spines.values(): spine.set_edgecolor("#2d3748")

plt.tight_layout()
plt.savefig("/Users/thomasmyles/dev/betting/src/nba_steals_modeling/research/notebooks/model_calibration_2025_26.png",
            dpi=120, bbox_inches="tight", facecolor="#0f1117")
plt.show()
print("Calibration table (2025-26):")
print(cal[["mid_p", "actual", "n"]].round(3).to_string())


In [ ]:
# Cell G: Threshold summary — which edge cutoff is consistent across folds?

if results:
    res_df = pd.DataFrame(results)
    pivot = res_df.pivot_table(index="threshold", columns="fold",
                               values=["roi", "n_bets"], aggfunc="first")
    # Flatten columns
    pivot.columns = [f"{v}_{c}" for v, c in pivot.columns]
    pivot = pivot.reset_index()

    print("=== ROI by edge threshold across walk-forward folds ===")
    print("Threshold | n 2024-25 | ROI 2024-25 | n 2025-26 | ROI 2025-26")
    print("-" * 65)
    for _, row in pivot.iterrows():
        thresh = row["threshold"]
        folds_rois = []
        for s in ["2024-25", "2025-26"]:
            n_col  = f"n_bets_{s}"
            r_col  = f"roi_{s}"
            if n_col in row and pd.notna(row[n_col]):
                folds_rois.append((int(row[n_col]), row[r_col]))
            else:
                folds_rois.append((0, float('nan')))
        print(f"  {thresh:>+6.0%}   "
              + "   ".join(f"n={n:>4} ROI={r*100:>+6.1f}%" for n, r in folds_rois))

    print()
    # Verdict
    for thresh in EDGE_THRESHOLDS:
        sub = res_df[res_df["threshold"] == thresh]
        positive_folds = (sub["roi"] > 0).sum()
        min_n = sub["n_bets"].min()
        verdict = "PASS" if positive_folds == len(sub) and min_n >= 100 else "FAIL"
        print(f"  Threshold {thresh:+.0%}: {positive_folds}/{len(sub)} folds positive, "
              f"min n={min_n} → {verdict}")
else:
    print("No results collected — check Cell E for errors.")


## GBM Model (LightGBM)

In [ ]:
# Cell H: LightGBM walk-forward — richer features, non-linear interactions

import lightgbm as lgb
from sklearn.metrics import brier_score_loss

# ── Extra features on top of Cell D's dc ──────────────────────────────────
dc2 = df_clean.sort_values(["PLAYER_NAME", "season", "game_date"]).copy()

def roll_shift(series, window):
    return series.shift(1).rolling(window, min_periods=max(1, window // 2)).mean()

grp2 = dc2.groupby(["PLAYER_NAME", "season"])

# Steal rolling features
dc2["stl_roll5"]            = grp2["STL"].transform(lambda s: roll_shift(s, 5))
dc2["stl_roll10"]           = grp2["STL"].transform(lambda s: roll_shift(s, 10))
dc2["stl_roll20"]           = grp2["STL"].transform(lambda s: roll_shift(s, 20))
dc2["min_roll10"]           = grp2["MIN"].transform(lambda s: roll_shift(s, 10))
dc2["min_roll20"]           = grp2["MIN"].transform(lambda s: roll_shift(s, 20))
dc2["stl_per_min_roll10"]   = dc2["stl_roll10"] / dc2["min_roll10"].replace(0, float("nan"))

# % of last 10 games where player got 2+ steals (pre-game hit rate)
dc2["stl_hit_rate_roll10"]  = grp2["y_under"].transform(
    lambda s: s.shift(1).rolling(10, min_periods=5).mean()
)

# Steal volatility — high std means unpredictable
dc2["stl_std_roll10"]       = grp2["STL"].transform(
    lambda s: s.shift(1).rolling(10, min_periods=5).std()
)

# Days rest (0 = back-to-back)
dc2["game_date_dt"] = pd.to_datetime(dc2["game_date"])
dc2["days_rest"] = (
    grp2["game_date_dt"]
    .transform(lambda s: s.diff().dt.days.shift(1))
    .clip(upper=7)
    .fillna(3)
)

# Games played in season so far
dc2["games_played"] = grp2.cumcount()

# Market-implied P(under)
dc2["market_p_under"] = 1 - dc2["p_mkt_dv"]

GBM_FEATURES = [
    "stl_roll5", "stl_roll10", "stl_roll20",
    "min_roll10", "min_roll20",
    "stl_per_min_roll10",
    "stl_hit_rate_roll10", "stl_std_roll10",
    "days_rest", "games_played",
    "market_p_under",
]

dc2_model = dc2.dropna(subset=GBM_FEATURES).copy()
print(f"Rows for GBM: {len(dc2_model):,} / {len(dc2):,}")
print(f"Base rate P(under): {dc2_model['y_under'].mean():.3f}")

# ── Walk-forward ───────────────────────────────────────────────────────────
SEASONS_ORDERED = ["2023-24", "2024-25", "2025-26"]
EDGE_THRESHOLDS = [0.03, 0.05, 0.07, 0.10]

gbm_params = dict(
    n_estimators=300,
    learning_rate=0.03,
    max_depth=4,
    num_leaves=15,
    min_child_samples=30,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    verbose=-1,
)

gbm_results = []

for test_idx in range(1, len(SEASONS_ORDERED)):
    train_seasons = SEASONS_ORDERED[:test_idx]
    test_season   = SEASONS_ORDERED[test_idx]

    train = dc2_model[dc2_model["season"].isin(train_seasons)].copy()
    test  = dc2_model[dc2_model["season"] == test_season].copy()

    model = lgb.LGBMClassifier(**gbm_params)
    model.fit(train[GBM_FEATURES], train["y_under"])

    test = test.copy()
    test["model_p_under"] = model.predict_proba(test[GBM_FEATURES])[:, 1]
    test["edge"] = test["model_p_under"] - test["market_p_under"]

    brier_model  = brier_score_loss(test["y_under"], test["model_p_under"])
    brier_market = brier_score_loss(test["y_under"], test["market_p_under"])

    print(f"=== Fold: train {train_seasons} → test {test_season} ===")
    print(f"  n_train={len(train):,}  n_test={len(test):,}")
    print(f"  Brier GBM: {brier_model:.5f}  |  Brier market: {brier_market:.5f}  |  "
          f"{'GBM WINS' if brier_model < brier_market else 'market wins'}")
    print()
    print(f"  {'Edge thresh':>12}  {'n_bets':>8}  {'hit_rate':>10}  {'ROI':>8}  {'PnL':>8}")
    print(f"  {'-'*54}")

    flat_roi = test["pnl"].mean()
    flat_hr  = test["y_under"].mean()
    print(f"  {'flat rule':>12}  {len(test):>8}  {flat_hr*100:>9.1f}%  {flat_roi*100:>7.1f}%  {test['pnl'].sum():>7.1f}u")

    for thresh in EDGE_THRESHOLDS:
        bets = test[test["edge"] >= thresh]
        if len(bets) < 10:
            print(f"  {thresh:>+12.0%}  {'<10':>8}  {'—':>10}  {'—':>8}  {'—':>8}")
            continue
        roi = bets["pnl"].mean()
        hr  = bets["y_under"].mean()
        pnl = bets["pnl"].sum()
        print(f"  {thresh:>+12.0%}  {len(bets):>8}  {hr*100:>9.1f}%  {roi*100:>7.1f}%  {pnl:>7.1f}u")
        gbm_results.append({"fold": test_season, "threshold": thresh,
                            "n_bets": len(bets), "roi": roi, "hit_rate": hr})
    print()

    # Feature importance
    fi = pd.Series(model.feature_importances_, index=GBM_FEATURES)
    fi = fi.sort_values(ascending=False)
    print("  Feature importance (gain):")
    for feat, imp in fi.items():
        bar = "#" * int(imp / fi.max() * 20)
        print(f"    {feat:30s}  {imp:5.0f}  {bar}")
    print()

# ── Cross-fold summary ──────────────────────────────────────────────────────
print("=== GBM threshold summary ===")
print(f"  {'Threshold':>10}  {'2024-25 n':>10}  {'2024-25 ROI':>12}  {'2025-26 n':>10}  {'2025-26 ROI':>12}  {'Verdict':>8}")
print("  " + "-" * 72)
for thresh in EDGE_THRESHOLDS:
    rows = [r for r in gbm_results if r["threshold"] == thresh]
    if not rows:
        print(f"  {thresh:>+10.0%}  {'—':>10}  {'—':>12}  {'—':>10}  {'—':>12}  {'—':>8}")
        continue
    by_fold = {r["fold"]: r for r in rows}
    f1 = by_fold.get("2024-25", {})
    f2 = by_fold.get("2025-26", {})
    pos_folds = sum(1 for r in rows if r["roi"] > 0)
    min_n = min(r["n_bets"] for r in rows)
    verdict = "PASS" if pos_folds == len(rows) and min_n >= 100 else ("WEAK" if pos_folds == len(rows) else "FAIL")
    print(f"  {thresh:>+10.0%}  {f1.get('n_bets',0):>10}  {f1.get('roi',0)*100:>+11.1f}%  "
          f"{f2.get('n_bets',0):>10}  {f2.get('roi',0)*100:>+11.1f}%  {verdict:>8}")


## Verdict

| | |
|---|---|
| **n_bets** | TBD |
| **Overall hit rate** | TBD |
| **Overall ROI** | TBD |
| **ROI 2023-24** | TBD |
| **ROI 2024-25** | TBD |
| **ROI 2025-26** | TBD |
| **Best segment** | TBD |

### Is this real?
- If ROI > 0 in 2/3 seasons: calibration gap is real and consistent — worth a v3 with model on top
- If ROI < 0 in 2/3 seasons: calibration gap is a data artifact or driven by extreme odds
- Key check: ROI at under_odds < -150 (market confident unders) — if still negative, the gap is illusory

### Next step
If positive: `v3_steals_1_5_under_model.ipynb` — add features on top of the flat-rule universe to find highest-confidence spots